# Analyzing Data Science Task to LMs

In [1]:
# imports
import yaml
import json
from stat_genie.blade_pipeline.baselines.config import MultiRunConfig
from stat_genie.blade_pipeline.baselines.multirun import multirun_llm
from stat_genie.blade_pipeline.additions.prompt.prompt import PromptGenerator
import os
from os.path import join
from pathlib import Path
from stat_genie.blade_pipeline.additions.analysis.get_model_output import get_model_output
from stat_genie.blade_pipeline.additions.perturbations.feature_names import FeaturePerturbation
from stat_genie.blade_pipeline.additions.analysis.fix_code import check_and_fix_code
import importlib.util
import sys
import pandas as pd
from stat_genie.blade_pipeline.baselines.multirun import _format_cvars_for_prompt
from stat_genie.blade_pipeline.additions.analysis.conclusion import write_final_answer_code, make_conclusion

In [2]:
### set config parameters

# set up config object
llm_provider = "openai"
llm_model = "gpt-5-mini"
llm_config = yaml.safe_load(open("../../config/llm_config.yml"))
llm_config["provider"] = llm_provider
llm_config["model"] = llm_model
llm_eval_config = llm_config

# set rest of parameters
output_dir = "analysis3_output"
run_dataset = "hurricane"
use_agent = False
use_data_desc = True
num_runs=3
use_code_cache=False

In [3]:
# the MultiRunConfig object is how BLADE standardizes experiment configuration
single_run_config = MultiRunConfig(llm=llm_config,
                llm_eval=llm_eval_config,
                output_dir=output_dir,
                run_dataset=run_dataset,
                use_agent=use_agent,
                use_data_desc=use_data_desc,
                num_runs=num_runs,
                use_code_cache=use_code_cache,
                fix_code=True,
)

In [4]:
# version #2 of the analysis applies a feature perturbation
# that masks the variable names and shuffles their order.
feature_perturbation = FeaturePerturbation(shuffle_names=True,
                                           shuffle_names_seed=123)

In [5]:
# prevent cache use
single_run_config.llm.use_cache = False
single_run_config.llm_eval.use_cache = False

In [6]:
### run the experiment
# get the features and transform/model code
multirun_llm(single_run_config, feature_perturbation=feature_perturbation)

[2025-12-04 11:20:27.58][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:20:28.04][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:20:28.28][llm.py:109 - stat_genie.blade_pipeline.llms.llm:generate][PROMPT] Sending prompt from <class 'stat_genie.blade_pipeline.baselines.lm.gen_analysis.GenAnalysisLM'>
===================[[system]]===================
You are an AI Data Analysis Assistant who is an expert at writing an end-to-end scientific analysis given a research question and a dataset. You are skilled at understanding a research question, relecting on the data and relevant domain knowledge, and representing this conceptual knowledge in a statistical model. Key to this modeling process is formalizing the conceptual model, which inc

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[2025-12-04 11:24:42.69][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:24:43.04][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 11:25:28.95][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  45.91 seconds
[2025-12-04 11:25:28.96][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 11:25:29.13][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:25:29.64][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 11:26:19.89][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  50.24 

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[2025-12-04 11:27:14.78][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


...

[2025-12-04 11:28:41.41][multirun.py:187 - stat_genie.blade_pipeline.baselines.multirun:__save_results][INFO] Fixed code for analysis 2 in 0 iteration(s)


/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[2025-12-04 11:28:41.96][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:28:42.78][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 11:29:20.02][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  37.24 seconds
[2025-12-04 11:29:20.03][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 11:29:20.07][multirun.py:236 - stat_genie.blade_pipeline.baselines.multirun:__save_results][INFO] Wrote final answer extraction code for analysis 0. It required 0 iteration(s) to be correct.
[2025-12-04 11:29:20.20][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:29:20.45][base.py:60 - stat_gen

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


[2025-12-04 11:29:32.79][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:29:33.26][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-12-04 11:30:12.66][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  39.41 seconds
[2025-12-04 11:30:12.67][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-12-04 11:30:12.71][multirun.py:236 - stat_genie.blade_pipeline.baselines.multirun:__save_results][INFO] Wrote final answer extraction code for analysis 1. It required 0 iteration(s) to be correct.
[2025-12-04 11:30:12.84][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.
[2025-12-04 11:30:13.09][base.py:60 - stat_gen